# Object_Detection Phase 5 — YOLOv11 파인튜닝

**목적**: AI Hub 음식이미지 데이터 전체(800종)로 yolo11s.pt 파인튜닝 → 한식/지방특산물 직접 탐지

**대상 데이터셋**: AI Hub '비전영역, 음식이미지 및 정보소개 텍스트 데이터' (dataSetSn=71564)  
- 총 232,087장 / 800종 / 4대분류 (특수외식 35% / 일반외식배달 34.7% / 끼니대체 26.4% / 음료차류 3.9%)

**결과물**: `best.pt` → 로컬 `Object_Detection/models/korean_food_v1_best.pt`

---

### 파일 키 목록

| 파일 | 키 | 크기 | 분류 |
|------|----|------|------|
| TS.z01 | 502331 | 100 GB | 학습 이미지 분할 1 |
| TS.z02 | 502332 | 100 GB | 학습 이미지 분할 2 |
| TS.z03 | 502333 | 100 GB | 학습 이미지 분할 3 |
| TS.z04 | 502334 | 100 GB | 학습 이미지 분할 4 |
| TS.z05 | 502335 | 100 GB | 학습 이미지 분할 5 |
| TS.z06 | 502336 | 100 GB | 학습 이미지 분할 6 |
| TS.z07 | 502337 | 100 GB | 학습 이미지 분할 7 |
| TS.zip | 502338 | 39.51 GB | 학습 이미지 분할 마지막 |
| **TL.zip** | **502339** | **142 MB** | **학습 라벨 JSON ← 즉시 다운로드** |
| VS.zip | 502340 | 88.80 GB | 검증 이미지 (선택) |
| **VL.zip** | **502341** | **17.77 MB** | **검증 라벨 JSON ← 즉시 다운로드** |

### 실행 순서
1. 런타임 → GPU(A100) 설정
2. Step 0: 환경 설정
3. Step 0.5: aihubshell 설치 → API 키 입력 → TL+VL 다운로드 → 학습 이미지 분할 처리
4. Step 1~7: 변환 → 학습 → 검증 → best.pt 저장

## Step 0 — 환경 설정 및 Drive 마운트

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
print('GPU 이름:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT   = '/content/drive/MyDrive/LGHellovision/Project 02/Object Detection'
DATASET_DIR  = f'{DRIVE_ROOT}/aihub_food'
FINETUNE_DIR = f'{DRIVE_ROOT}/finetune_dataset'
RUNS_DIR     = f'{DRIVE_ROOT}/runs'
LOCAL_TMP    = '/content/aihub_tmp'   # 공백 없는 Colab 로컬 (aihubshell 다운로드용)

for d in [
    DATASET_DIR, FINETUNE_DIR, RUNS_DIR, LOCAL_TMP,
    f'{FINETUNE_DIR}/train/images', f'{FINETUNE_DIR}/train/labels',
    f'{FINETUNE_DIR}/val/images',   f'{FINETUNE_DIR}/val/labels',
]:
    os.makedirs(d, exist_ok=True)

print('Drive 마운트 완료')
print(f'DATASET_DIR:  {DATASET_DIR}')
print(f'FINETUNE_DIR: {FINETUNE_DIR}')
print(f'RUNS_DIR:     {RUNS_DIR}')
print(f'LOCAL_TMP:    {LOCAL_TMP}')
!df -h /content | tail -1
!df -h /content/drive/MyDrive | tail -1

In [ ]:
!pip install ultralytics -q
print('ultralytics 설치 완료')

## Step 0.5 — AI Hub Shell 다운로드

> AI Hub는 REST API 직접 호출 불가 → **aihubshell** 전용 툴 사용  
> 마이페이지 → API 키 발급 후 아래 셀에 입력

| 파일 | 크기 | 전략 |
|------|------|------|
| TL.zip + VL.zip | ~160 MB | 즉시 Drive 저장 |
| TS.z01~TS.zip | 각 100 GB | 로컬 1개씩 받고 이미지만 Drive 이동 후 삭제 |
| VS.zip | 88.80 GB | 선택 다운로드 |

In [ ]:
import os
import subprocess
import shutil
from pathlib import Path

# ── aihubshell 설치 ──────────────────────────────────────────────────────────
!curl -o /usr/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do -s
!chmod +x /usr/bin/aihubshell
print('aihubshell 설치 완료')

# ── 인증 정보 입력 ─────────────────────────────────────────────────────────────
# 방법 1: API 키  (마이페이지 → 오픈 API → API 키 발급)
# 방법 2: ID/PW  (API 키가 없거나 502 오류 시)
# ↓ 사용할 방법만 입력, 나머지는 Enter 스킵
AIHUB_API_KEY = input('AI Hub API 키 (없으면 Enter 스킵): ').strip()
AIHUB_ID      = ''
AIHUB_PW      = ''
if not AIHUB_API_KEY:
    AIHUB_ID  = input('AI Hub 이메일: ').strip()
    AIHUB_PW  = input('AI Hub 비밀번호: ').strip()

AUTH_MODE = 'apikey' if AIHUB_API_KEY else 'idpw'
print(f'인증 방식: {AUTH_MODE}')

# ── 상수 ──────────────────────────────────────────────────────────────────────
DATASET_KEY = 71564

FILE_KEYS = {
    'TL.zip': 502339,
    'VL.zip': 502341,
    'TS.z01': 502331,
    'TS.z02': 502332,
    'TS.z03': 502333,
    'TS.z04': 502334,
    'TS.z05': 502335,
    'TS.z06': 502336,
    'TS.z07': 502337,
    'TS.zip': 502338,
    'VS.zip': 502340,
}


def _build_auth_args():
    """인증 방식에 따라 aihubshell 인수 반환"""
    if AUTH_MODE == 'apikey':
        return ['-aihubapikey', AIHUB_API_KEY]
    else:
        return ['-aihubid', AIHUB_ID, '-aihubpw', AIHUB_PW]


def aihub_download(file_keys: list, dest_dir: str):
    """
    aihubshell 다운로드 래퍼.
    경로 공백 문제 우회: LOCAL_TMP(공백 없음)에 먼저 받고 → dest_dir 로 이동.
    aihubshell이 압축 해제까지 자동 처리.
    """
    os.makedirs(dest_dir, exist_ok=True)
    key_str = ','.join(str(k) for k in file_keys)

    cmd = (
        ['aihubshell', '-mode', 'd',
         '-datasetkey', str(DATASET_KEY),
         '-filekey', key_str]
        + _build_auth_args()
    )

    print(f'[1/2] 로컬 다운로드 → {LOCAL_TMP}  (filekey={key_str})')
    result = subprocess.run(cmd, cwd=LOCAL_TMP, capture_output=True, text=True)

    print(result.stdout[-2000:] if result.stdout else '(stdout 없음)')
    if result.stderr:
        print('stderr:', result.stderr[-500:])
    if result.returncode != 0:
        raise RuntimeError(f'aihubshell 실패 returncode={result.returncode}')

    local_files = [f for f in Path(LOCAL_TMP).rglob('*') if f.is_file()]
    print(f'\n로컬 파일: {len(local_files)}개')
    for f in local_files[:10]:
        print(f'  {f.relative_to(LOCAL_TMP)}  ({f.stat().st_size/1024/1024:.1f} MB)')
    if not local_files:
        raise RuntimeError('다운로드 파일 없음 — stdout/stderr 확인')

    print(f'\n[2/2] Drive로 이동 → {dest_dir}')
    for f in local_files:
        dst = Path(dest_dir) / f.relative_to(LOCAL_TMP)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(f), str(dst))
    print('✅ 완료')


print('설정 완료')

In [ ]:
# ── 인증 테스트 (목록 조회) ────────────────────────────────────────────────────
result = subprocess.run(
    ['aihubshell', '-mode', 'l', '-datasetkey', str(DATASET_KEY)]
    + _build_auth_args(),
    capture_output=True, text=True
)
print(result.stdout[:3000])
if result.stderr:
    print('stderr:', result.stderr[-300:])

# 파일 키 번호가 stdout에 있으면 목록 조회 성공
tree_ok = any(str(k) in result.stdout for k in FILE_KEYS.values())

if tree_ok:
    print('✅ 목록 조회 성공 — 다음 셀 진행')
    if AUTH_MODE == 'apikey':
        print()
        print('⚠️  단, 다운로드는 ID/PW 방식이 필요할 수 있습니다.')
        print('   다운로드 502 오류 발생 시:')
        print('   → cell-6 재실행 → API 키 Enter 스킵 → 이메일+비밀번호 입력')
else:
    print('❌ 목록 조회 실패')
    print('  → cell-6 재실행 → API 키 Enter 스킵 → ID/PW 방식으로 재시도')

In [ ]:
import json

# TL.zip(학습 라벨 142MB) + VL.zip(검증 라벨 18MB) 즉시 다운로드
aihub_download(
    file_keys=[FILE_KEYS['TL.zip'], FILE_KEYS['VL.zip']],
    dest_dir=DATASET_DIR,
)

# 다운로드 결과 확인 — aihubshell이 만든 폴더 구조 그대로 탐색
all_jsons = list(Path(DATASET_DIR).rglob('*.json'))
print(f'\nJSON 파일 수: {len(all_jsons):,}개')

if all_jsons:
    print(f'샘플 경로: {all_jsons[0]}')
    with open(all_jsons[0], encoding='utf-8') as f:
        s = json.load(f)
    print(f'최상위 키: {list(s.keys())}')
    if 'data' in s:
        print(f'data 하위: {list(s["data"].keys())}')

# LABEL_DIR: JSON이 있는 최상위 공통 경로로 자동 설정
LABEL_DIR = str(all_jsons[0].parent) if all_jsons else DATASET_DIR
# 더 정확히는 DATASET_DIR 전체를 스캔
LABEL_DIR = DATASET_DIR
print(f'\nLABEL_DIR: {LABEL_DIR}')

In [ ]:
# 학습 이미지 분할 파일 목록
TRAIN_IMG_DIR = f'{DATASET_DIR}/train_images'
os.makedirs(TRAIN_IMG_DIR, exist_ok=True)

TRAIN_PARTS = [
    ('TS.z01', 502331),
    ('TS.z02', 502332),
    ('TS.z03', 502333),
    ('TS.z04', 502334),
    ('TS.z05', 502335),
    ('TS.z06', 502336),
    ('TS.z07', 502337),
    ('TS.zip', 502338),
]

!df -h /content | tail -1
!df -h /content/drive/MyDrive | tail -1

In [ ]:
def process_one_part(file_name: str, file_key: int):
    """
    분할 파일 1개: LOCAL_TMP에 다운로드+해제 → 이미지만 Drive 이동 → 로컬 삭제
    """
    existing = len(list(Path(TRAIN_IMG_DIR).rglob('*.jpg')))
    print(f'현재 Drive train_images: {existing:,}장')

    aihub_download(file_keys=[file_key], dest_dir=f'{LOCAL_TMP}/{file_name}_out')

    extract_dir = Path(f'{LOCAL_TMP}/{file_name}_out')
    imgs = (list(extract_dir.rglob('*.jpg')) +
            list(extract_dir.rglob('*.jpeg')) +
            list(extract_dir.rglob('*.png')))
    print(f'추출 이미지: {len(imgs):,}장 → Drive 이동')

    for img in imgs:
        dst = Path(TRAIN_IMG_DIR) / img.name
        if not dst.exists():
            shutil.move(str(img), str(dst))

    shutil.rmtree(str(extract_dir), ignore_errors=True)

    total = len(list(Path(TRAIN_IMG_DIR).rglob('*.jpg')))
    print(f'✅ {file_name} 완료 | Drive 누적: {total:,}장')
    !df -h /content | tail -1

print('process_one_part 준비 완료')

In [ ]:
# 분할 파일 1개씩 처리 — PART_INDEX를 0→7 순으로 바꿔가며 실행
PART_INDEX = 0   # ← 여기만 바꾸면 됨

file_name, file_key = TRAIN_PARTS[PART_INDEX]
print(f'처리: {file_name} [{PART_INDEX+1}/{len(TRAIN_PARTS)}]')
process_one_part(file_name, file_key)

In [ ]:
# [선택] VS.zip — 검증 이미지 (88.80 GB)
DOWNLOAD_VS = False   # Drive 여유 90GB 이상일 때 True로 변경

if DOWNLOAD_VS:
    aihub_download(
        file_keys=[FILE_KEYS['VS.zip']],
        dest_dir=f'{DATASET_DIR}/val_images',
    )
else:
    print('VS.zip 스킵 — 검증은 학습 데이터 80:20 분리로 대체')

In [ ]:
# 데이터 준비 현황 요약
train_imgs  = len(list(Path(TRAIN_IMG_DIR).rglob('*.jpg'))) if os.path.exists(TRAIN_IMG_DIR) else 0
label_jsons = len(list(Path(LABEL_DIR).rglob('*.json')))

print('=== 현황 ===')
print(f'라벨 JSON: {label_jsons:,}개')
print(f'학습 이미지: {train_imgs:,}장')

if label_jsons == 0:
    print('⚠️  TL.zip 다운로드 필요')
if train_imgs == 0:
    print('⚠️  학습 이미지 없음 — PART_INDEX 0부터 실행')

## Step 1 — 전체 클래스 수집

> AI Hub JSON의 `food_type.fc` 를 전부 스캔 → 800종 클래스 목록 자동 구성

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

def collect_all_classes(label_dir: str):
    json_files = list(Path(label_dir).rglob('*.json'))
    print(f'JSON 파일: {len(json_files):,}개')

    class_count = defaultdict(int)
    errors = 0
    for jf in json_files:
        try:
            with open(jf, encoding='utf-8') as f:
                d = json.load(f)
            fc = d.get('data', {}).get('food_type', {}).get('fc', '').strip()
            if fc:
                class_count[fc] += 1
        except Exception:
            errors += 1

    sorted_cls = sorted(class_count.items(), key=lambda x: -x[1])
    names = [c for c, _ in sorted_cls]
    print(f'클래스 수: {len(names)}종 / 파싱 오류: {errors}개')
    print('\n상위 20개:')
    for c, n in sorted_cls[:20]:
        print(f'  {c}: {n}개')
    return names, dict(class_count)


ALL_CLASSES, CLASS_COUNT = collect_all_classes(LABEL_DIR)
CLASS_TO_ID = {c: i for i, c in enumerate(ALL_CLASSES)}
print(f'\nnc = {len(ALL_CLASSES)}')

## Step 2 — JSON → YOLO 포맷 변환

> 변환식: `cx=(x+bw/2)/img_w`, `cy=(y+bh/2)/img_h`  
> 클래스 필터링 없음 — bbox 오류/이미지 없음만 skip

In [ ]:
import shutil
import random

def convert_one(json_path):
    try:
        with open(json_path, encoding='utf-8') as f:
            raw = json.load(f)
        d = raw.get('data', {})

        fc = d.get('food_type', {}).get('fc', '').strip()
        if not fc or fc not in CLASS_TO_ID:
            return None

        ii = d.get('image_info', {})
        iw, ih = float(ii.get('width', 0)), float(ii.get('height', 0))
        fname = ii.get('file_name', '')
        if iw <= 0 or ih <= 0 or not fname:
            return None

        a = d.get('2d_annotation', {})
        x, y = float(a.get('x', 0)), float(a.get('y', 0))
        bw, bh = float(a.get('width', 0)), float(a.get('height', 0))
        if bw <= 0 or bh <= 0:
            return None

        cx, cy = (x + bw/2)/iw, (y + bh/2)/ih
        nw, nh = bw/iw, bh/ih
        if not all(0.0 < v <= 1.0 for v in [cx, cy, nw, nh]):
            return None

        cid = CLASS_TO_ID[fc]
        lbl = f'{cid} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}'

        img = Path(json_path).parent / fname
        if not img.exists():
            img = Path(TRAIN_IMG_DIR) / fname
        if not img.exists():
            cands = list(Path(DATASET_DIR).rglob(fname))
            if not cands:
                return None
            img = cands[0]

        return cid, lbl, str(img), fc
    except Exception:
        return None


def build_full_dataset(label_dir, output_dir, val_ratio=0.2):
    json_files = list(Path(label_dir).rglob('*.json'))
    print(f'변환 대상: {len(json_files):,}개')

    records = defaultdict(list)
    skipped = 0
    for i, jf in enumerate(json_files):
        if i % 10000 == 0:
            print(f'  {i:,}/{len(json_files):,}', end='\r')
        r = convert_one(str(jf))
        if r is None:
            skipped += 1
        else:
            _, lbl, img, cls = r
            records[cls].append((lbl, img))

    total = sum(len(v) for v in records.values())
    print(f'\n변환 성공: {total:,} / 실패: {skipped:,}')

    train_n = val_n = 0
    for cls, recs in records.items():
        random.shuffle(recs)
        sp = max(1, int(len(recs) * (1 - val_ratio)))
        for split, part in [('train', recs[:sp]), ('val', recs[sp:])]:
            for lbl, src in part:
                src = Path(src)
                dst_img = Path(output_dir) / split / 'images' / src.name
                dst_lbl = Path(output_dir) / split / 'labels' / (src.stem + '.txt')
                shutil.copy2(str(src), str(dst_img))
                dst_lbl.write_text(lbl + '\n', encoding='utf-8')
                if split == 'train': train_n += 1
                else: val_n += 1

    print(f'train: {train_n:,} / val: {val_n:,}')
    return train_n, val_n


train_count, val_count = build_full_dataset(LABEL_DIR, FINETUNE_DIR)

## Step 3 — data.yaml 생성

In [ ]:
import yaml

yaml_path = f'{FINETUNE_DIR}/data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump({
        'train': f'{FINETUNE_DIR}/train/images',
        'val':   f'{FINETUNE_DIR}/val/images',
        'nc':    len(ALL_CLASSES),
        'names': ALL_CLASSES,
    }, f, allow_unicode=True, default_flow_style=False)

print(f'data.yaml 생성 완료 — nc={len(ALL_CLASSES)}')
for i, c in enumerate(ALL_CLASSES[:10]):
    print(f'  {i}: {c}')

## Step 4 — 파인튜닝 학습

> base: `yolo11s.pt` / conf=0.5 / iou=0.45 → `detection_config.yaml` 동일

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
print(f'학습 시작 — 클래스: {len(ALL_CLASSES)}종')

results = model.train(
    data=f'{FINETUNE_DIR}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    patience=20,
    project=RUNS_DIR,
    name='korean_food_v1',
    exist_ok=True,
    save=True,
    save_period=10,
    conf=0.001,
    iou=0.45,
    max_det=100,
)

print(f'학습 완료 — best.pt: {RUNS_DIR}/korean_food_v1/weights/best.pt')

## Step 5 — 검증

In [ ]:
from ultralytics import YOLO
import pandas as pd

model_ft = YOLO(f'{RUNS_DIR}/korean_food_v1/weights/best.pt')
metrics = model_ft.val(data=f'{FINETUNE_DIR}/data.yaml', device=0, conf=0.5, iou=0.45)

print(f'mAP@0.5:   {metrics.box.map50:.4f}  (목표 ≥ 0.60)')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

KEY_CLASSES = [
    '굴비', '조기', '대게', '전복', '장어', '한우', '흑돼지',
    '비빔밥', '김치찌개', '된장찌개', '삼겹살', '불고기',
    '사과', '딸기', '수박', '감귤',
]
if hasattr(metrics.box, 'ap_class_index'):
    rows = []
    for i, ci in enumerate(metrics.box.ap_class_index):
        nm = ALL_CLASSES[int(ci)] if int(ci) < len(ALL_CLASSES) else str(ci)
        if nm in KEY_CLASSES:
            rows.append({'클래스': nm, 'AP@0.5': round(float(metrics.box.ap50[i]), 4)})
    if rows:
        print(pd.DataFrame(rows).sort_values('AP@0.5', ascending=False).to_string(index=False))

print('\n✅ 합격' if metrics.box.map50 >= 0.60 else '\n⚠️  미달')

## Step 6 — A/B 비교

In [ ]:
from ultralytics import YOLO
import glob
from collections import Counter

val_imgs = glob.glob(f'{FINETUNE_DIR}/val/images/*.jpg')[:20]
if not val_imgs:
    print('val 이미지 없음')
else:
    base  = YOLO('yolo11s.pt')
    ft    = YOLO(f'{RUNS_DIR}/korean_food_v1/weights/best.pt')
    bc, fc_cnt = Counter(), Counter()
    for img in val_imgs:
        for p in base(img, conf=0.5, verbose=False):
            for b in p.boxes: bc[p.names[int(b.cls)]] += 1
        for p in ft(img, conf=0.5, verbose=False):
            for b in p.boxes: fc_cnt[p.names[int(b.cls)]] += 1

    print('기존 yolo11s.pt 상위 10:')
    for l, n in bc.most_common(10): print(f'  {l}: {n}')
    print('\n파인튜닝 best.pt 상위 10:')
    for l, n in fc_cnt.most_common(10): print(f'  {l}: {n}')

    KEY_SET = set(KEY_CLASSES)
    bk = sum(n for l,n in bc.items() if l in KEY_SET)
    fk = sum(n for l,n in fc_cnt.items() if l in KEY_SET)
    print(f'\n주요 클래스 탐지: 기존 {bk}건 → 파인튜닝 {fk}건 (+{fk-bk})')
    print('✅ 효과 확인' if fk > bk else '⚠️  효과 미미')

## Step 7 — best.pt 로컬 적용

In [ ]:
import os
best = f'{RUNS_DIR}/korean_food_v1/weights/best.pt'
if os.path.exists(best):
    print(f'✅ best.pt — {os.path.getsize(best)/1024/1024:.1f} MB')
    print()
    print('1. Drive에서 best.pt 다운로드')
    print(f'   {best}')
    print()
    print('2. 로컬 저장')
    print('   Object_Detection/models/korean_food_v1_best.pt')
    print()
    print('3. config/detection_config.yaml 수정 (한 줄)')
    print('   name: models/korean_food_v1_best.pt')
    print()
    print('4. python -m pytest tests/test_phase1_setup.py -v  → 13/13 PASS')
    print()
    print('⚠️  models/*.pt → .gitignore 대상, 커밋 금지')
else:
    print('⚠️  best.pt 없음 — Step 4 완료 후 재실행')